## 🧱 1. Setup & Install Dependencies

In [1]:
!pip install pdfforms pytesseract pdfplumber llama-index mistralai openai
!apt-get install poppler-utils tesseract-ocr


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 578.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.1/374.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.8/266.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.0/89.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.2/304.

In [ ]:
!pip install google-generativeai


## 📁 2. Mount Google Drive (storing PDFs there)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 🧾 3. OCR the Referral Package (via Mistral / pytesseract fallback)

In [ ]:
from PIL import Image
import pytesseract
import fitz  # PyMuPDF

def extract_text_from_scanned_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    extracted_text = []
    for page_num in range(len(doc)):
        pix = doc.load_page(page_num).get_pixmap()
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        text = pytesseract.image_to_string(img)
        extracted_text.append((page_num + 1, text))
    return extracted_text

ref_text = extract_text_from_scanned_pdf("/content/drive/MyDrive/Automate_Insurance_Claims/Input Data/Adbulla/referral_package.pdf")


## 🧠 4. Extract Information (e.g., using Gemini, LlamaParse, or Rule-Based)

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SimpleNodeParser
from llama_index.llms.openai import OpenAI

reader = SimpleDirectoryReader(input_files=["referral_text.txt"])
nodes = SimpleNodeParser().get_nodes_from_documents(reader.load_data())
llm = OpenAI(model="gpt-4", temperature=0)

# Use a prompt to extract structured data
query = """
Extract the following fields:
- Patient name
- DOB
- Drug name
- Prescribing physician
- Diagnosis (ICD-10 if available)
Return as JSON.
"""
response = llm.complete(prompt=query + "\n" + "\n".join(text for _, text in ref_text))
print(response.text)


## 📝 5. Fill the PA Form (using pdfforms)

In [ ]:
from pdfforms import fill_pdf
import json

# Load field mapping from JSON response
extracted = json.loads(response.text)

# Map to form field names manually or via heuristics
field_map = {
    "patient_name": extracted.get("Patient name", ""),
    "dob": extracted.get("DOB", ""),
    "drug": extracted.get("Drug name", ""),
    "physician": extracted.get("Prescribing physician", ""),
    "diagnosis": extracted.get("Diagnosis", "")
}

fill_pdf(
    input_pdf="/content/drive/MyDrive/Automate_Insurance_Claims/Input Data/Adbulla/PA.pdf",
    output_pdf="filled_PA_Patient_A.pdf",
    data=field_map
)


## 📄 6. Generate Missing Fields Report

In [ ]:
required_fields = ["patient_name", "dob", "drug", "physician", "diagnosis"]
missing = [f for f in required_fields if not field_map.get(f)]

with open("missing_fields_Patient_A.md", "w") as f:
    f.write("# Missing Fields Report - Patient A\n\n")
    for field in missing:
        f.write(f"- [ ] {field.replace('_', ' ').title()} — missing\n")
